In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [2]:
GEN_1_RANDOM = CF_OUTPUTS / "gen1_models_explainers_constraints" / "random_search_exp"
GEN_1_RANDOM.is_dir()

True

In [3]:
batch_data = GEN_1_RANDOM / "RF_highthres_2026-04-21" / "annotated.csv"
batch_df = pd.read_csv(batch_data)

### set PD options

In [4]:
pd.set_option("display.max_rows", None)

In [5]:
# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [6]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

In [7]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.34,,,,0.0633,
1,0,cf_1,,,,,,,17.1346,,,0.0115,1,False,0.0633,0.0467
2,0,cf_2,,,,,,,30.1683,,,0.0695,1,False,0.0633,0.1133
3,0,cf_3,7,,,,,,16.9519,,,0.1376,2,False,0.0633,0.0833
4,0,cf_4,,6,,,,,27.877,,,0.149,2,False,0.0633,0.1967
5,0,cf_5,,4,1,,,,,,,0.1562,2,True,0.0633,0.03
6,0,cf_6,,2,,,,1,,,,0.1562,2,False,0.0633,0.11
7,0,cf_7,,,,,,1,37.0705,,,0.2374,2,False,0.0633,0.09
8,0,cf_8,5,,,,,,29.8874,,,0.1094,2,False,0.0633,0.06
9,0,cf_9,,,,,,,33.6511,,,0.0911,1,False,0.0633,0.2233


In [8]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation
# prefers valid CFs without violations, and lowest Gower
single_gower_df = select_one_cf_per_query(batch_df, selector="gower")
single_gower_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.34,,,,0.0633,
9,0,cf_5,,4,1,,,,,,,0.1562,2,True,0.0633,0.03
1,1,xin,3,4,1,2,3,0,22.3757,0,0.33,,,,0.1467,
10,1,cf_2,,,,,,1,,,,0.125,1,True,0.1467,0.0733
2,2,xin,5,3,1,1,4,0,22.694,7,0.32,,,,0.1567,
11,2,cf_2,,,,,,,26.8578,,,0.0386,1,True,0.1567,0.07
3,3,xin,3,3,6,6,2,0,24.3418,1,0.34,,,,0.02,
12,3,cf_1,,,,,,,17.9625,4,,0.132,2,False,0.02,0.1867
4,4,xin,5,4,2,7,1,0,21.2585,3,0.33,,,,0.0267,
13,4,cf_5,4,2,,,,,,,,0.125,2,True,0.0267,0.01


In [9]:
# Random selection (Gower may pick only BMI variations)
single_random_df = select_one_cf_per_query(batch_df, selector="random")
single_random_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.34,,,,0.0633,
9,0,cf_5,,4,1,,,,,,,0.1562,2,True,0.0633,0.03
1,1,xin,3,4,1,2,3,0,22.3757,0,0.33,,,,0.1467,
10,1,cf_2,,,,,,1,,,,0.125,1,True,0.1467,0.0733
2,2,xin,5,3,1,1,4,0,22.694,7,0.32,,,,0.1567,
11,2,cf_8,,,,,1,1,,,,0.25,2,True,0.1567,0.02
3,3,xin,3,3,6,6,2,0,24.3418,1,0.34,,,,0.02,
12,3,cf_1,,,,,,,17.9625,4,,0.132,2,False,0.02,0.1867
4,4,xin,5,4,2,7,1,0,21.2585,3,0.33,,,,0.0267,
13,4,cf_5,4,2,,,,,,,,0.125,2,True,0.0267,0.01
